- Physical
- Imaginary
+ Sub Universes in Imaginary Space

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Tuple, Optional


class HyperbolicEmbedder(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.projection = nn.Linear(input_dim, input_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Exponential map projection into hyperbolic space.
        Uses Poincaré ball model approximation.
        """
        x = self.projection(x)
        norm = torch.norm(x, dim=-1, keepdim=True)
        return torch.tanh(norm) * (x / (norm + 1e-8))


class SphericalEmbedder(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.projection = nn.Linear(input_dim, input_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Project input to unit sphere surface.
        """
        x = self.projection(x)
        norm = torch.norm(x, dim=-1, keepdim=True)
        return x / (norm + 1e-8)


class FractalEmbedder(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.recursive_transform = nn.Sequential(
            nn.Linear(input_dim, input_dim * 2),
            nn.ReLU(),
            nn.Linear(input_dim * 2, input_dim),
            nn.Tanh()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Recursive transformation mimicking fractal-like embeddings.
        """
        return self.recursive_transform(x)


class GeometryAwarenessFunction(nn.Module):
    """
    A dynamic geometry awareness mechanism for adaptive data representation.

    Supports multiple geometric spaces and intelligent geometry selection.
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 64,
        geometry_types: Optional[List[str]] = None,
    ):
        """
        Initialize the Geometry Awareness Function.

        Args:
            input_dim (int): Dimensionality of input features
            hidden_dim (int): Hidden layer dimension for geometry classification
            geometry_types (List[str], optional): List of geometry types to consider
        """
        super().__init__()

        self.input_dim = input_dim

        # Geometry types
        if geometry_types is None:
            self.GEOMETRY_TYPES = ['euclidean', 'hyperbolic', 'spherical', 'fractal']
        else:
            self.GEOMETRY_TYPES = geometry_types
        num_geometries = len(self.GEOMETRY_TYPES)

        # Geometry detection network
        self.geometry_classifier = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, num_geometries)
            # Softmax will be applied in the forward method
        )

        # Embedding transformers for different geometries
        self.geometry_embedders = nn.ModuleDict({
            geom: self._create_embedder(geom) for geom in self.GEOMETRY_TYPES
        })

    def _create_embedder(self, geometry_type: str) -> nn.Module:
        """
        Create an embedder for the specified geometry type.

        Args:
            geometry_type (str): The geometry type

        Returns:
            nn.Module: The embedder module
        """
        if geometry_type == 'euclidean':
            return nn.Linear(self.input_dim, self.input_dim)
        elif geometry_type == 'hyperbolic':
            return HyperbolicEmbedder(self.input_dim)
        elif geometry_type == 'spherical':
            return SphericalEmbedder(self.input_dim)
        elif geometry_type == 'fractal':
            return FractalEmbedder(self.input_dim)
        else:
            raise ValueError(f"Unknown geometry type: {geometry_type}")

    def detect_geometry(self, x: torch.Tensor) -> torch.Tensor:
        """
        Classify the input's geometric characteristics.

        Args:
            x (torch.Tensor): Input feature tensor

        Returns:
            torch.Tensor: Geometry probabilities
        """
        geometry_logits = self.geometry_classifier(x)
        geometry_probs = F.softmax(geometry_logits, dim=-1)
        return geometry_probs

    def embed_in_geometry(
        self,
        x: torch.Tensor,
        geometry_probs: torch.Tensor
    ) -> torch.Tensor:
        """
        Embed input in the most appropriate geometry.

        Args:
            x (torch.Tensor): Input feature tensor
            geometry_probs (torch.Tensor): Probabilities for each geometry

        Returns:
            torch.Tensor: Geometry-specific embedding
        """
        # Compute embeddings for each geometry
        embeddings = torch.stack([
            self.geometry_embedders[geom](x)
            for geom in self.GEOMETRY_TYPES
        ], dim=1)  # Shape: [batch_size, num_geometries, embedding_dim]

        # Apply geometry probability weights
        weighted_embedding = torch.sum(
            embeddings * geometry_probs.unsqueeze(-1),
            dim=1
        )

        return weighted_embedding

    def forward(
        self,
        x: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass through the Geometry Awareness Function.

        Args:
            x (torch.Tensor): Input feature tensor

        Returns:
            Tuple[torch.Tensor, torch.Tensor]: Embedded features and geometry probabilities
        """
        # Detect geometry characteristics
        geometry_probs = self.detect_geometry(x)

        # Embed in selected geometries
        embedded_x = self.embed_in_geometry(x, geometry_probs)

        return embedded_x, geometry_probs


def test_geometry_awareness():
    """
    Demonstration and basic testing of the Geometry Awareness Function.
    """
    # Simulated input dimensions and batch size
    input_dim = 256
    batch_size = 64

    # Device configuration
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Initialize the Geometry Awareness Function
    geo_aware = GeometryAwarenessFunction(input_dim).to(device)

    # Generate random input data
    sample_input = torch.randn(batch_size, input_dim).to(device)

    # Process through Geometry Awareness Function
    geo_aware.eval()  # Set model to evaluation mode
    with torch.no_grad():
        embedded_features, geometry_probs = geo_aware(sample_input)

    print("Input Shape:", sample_input.shape)
    print("Embedded Features Shape:", embedded_features.shape)
    print("Geometry Probabilities (averaged over batch):")
    avg_geometry_probs = geometry_probs.mean(0)
    for geom, prob in zip(geo_aware.GEOMETRY_TYPES, avg_geometry_probs):
        print(f"{geom.capitalize()}: {prob.item():.4f}")


if __name__ == "__main__":
    test_geometry_awareness()

Input Shape: torch.Size([64, 256])
Embedded Features Shape: torch.Size([64, 256])
Geometry Probabilities (averaged over batch):
Euclidean: 0.2631
Hyperbolic: 0.2354
Spherical: 0.2765
Fractal: 0.2250


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Tuple, Optional
import numpy as np
import random
from torch.utils.data import DataLoader, TensorDataset


class HyperbolicEmbedder(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.projection = nn.Linear(input_dim, input_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Exponential map projection into hyperbolic space.
        Uses Poincaré ball model approximation.
        """
        x = self.projection(x)
        norm = torch.norm(x, dim=-1, keepdim=True)
        return torch.tanh(norm) * (x / (norm + 1e-8))


class SphericalEmbedder(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.projection = nn.Linear(input_dim, input_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Project input to unit sphere surface.
        """
        x = self.projection(x)
        norm = torch.norm(x, dim=-1, keepdim=True)
        return x / (norm + 1e-8)


class FractalEmbedder(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.recursive_transform = nn.Sequential(
            nn.Linear(input_dim, input_dim * 2),
            nn.ReLU(),
            nn.Linear(input_dim * 2, input_dim),
            nn.Tanh()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Recursive transformation mimicking fractal-like embeddings.
        """
        return self.recursive_transform(x)


class GeometryAwarenessFunction(nn.Module):
    """
    A dynamic geometry awareness mechanism for adaptive data representation.

    Supports multiple geometric spaces and intelligent geometry selection.
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 64,
        geometry_types: Optional[List[str]] = None,
    ):
        """
        Initialize the Geometry Awareness Function.

        Args:
            input_dim (int): Dimensionality of input features
            hidden_dim (int): Hidden layer dimension for geometry classification
            geometry_types (List[str], optional): List of geometry types to consider
        """
        super().__init__()

        self.input_dim = input_dim

        # Geometry types
        if geometry_types is None:
            self.GEOMETRY_TYPES = ['euclidean', 'hyperbolic', 'spherical', 'fractal']
        else:
            self.GEOMETRY_TYPES = geometry_types
        num_geometries = len(self.GEOMETRY_TYPES)

        # Geometry detection network
        self.geometry_classifier = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, num_geometries)
            # Softmax will be applied in the forward method
        )

        # Embedding transformers for different geometries
        self.geometry_embedders = nn.ModuleDict({
            geom: self._create_embedder(geom) for geom in self.GEOMETRY_TYPES
        })

    def _create_embedder(self, geometry_type: str) -> nn.Module:
        """
        Create an embedder for the specified geometry type.

        Args:
            geometry_type (str): The geometry type

        Returns:
            nn.Module: The embedder module
        """
        if geometry_type == 'euclidean':
            return nn.Linear(self.input_dim, self.input_dim)
        elif geometry_type == 'hyperbolic':
            return HyperbolicEmbedder(self.input_dim)
        elif geometry_type == 'spherical':
            return SphericalEmbedder(self.input_dim)
        elif geometry_type == 'fractal':
            return FractalEmbedder(self.input_dim)
        else:
            raise ValueError(f"Unknown geometry type: {geometry_type}")

    def detect_geometry(self, x: torch.Tensor) -> torch.Tensor:
        """
        Classify the input's geometric characteristics.

        Args:
            x (torch.Tensor): Input feature tensor

        Returns:
            torch.Tensor: Geometry probabilities
        """
        geometry_logits = self.geometry_classifier(x)
        geometry_probs = F.softmax(geometry_logits, dim=-1)
        return geometry_probs

    def embed_in_geometry(
        self,
        x: torch.Tensor,
        geometry_probs: torch.Tensor
    ) -> torch.Tensor:
        """
        Embed input in the most appropriate geometry.

        Args:
            x (torch.Tensor): Input feature tensor
            geometry_probs (torch.Tensor): Probabilities for each geometry

        Returns:
            torch.Tensor: Geometry-specific embedding
        """
        # Compute embeddings for each geometry
        embeddings = torch.stack([
            self.geometry_embedders[geom](x)
            for geom in self.GEOMETRY_TYPES
        ], dim=1)  # Shape: [batch_size, num_geometries, embedding_dim]

        # Apply geometry probability weights
        weighted_embedding = torch.sum(
            embeddings * geometry_probs.unsqueeze(-1),
            dim=1
        )

        return weighted_embedding

    def forward(
        self,
        x: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass through the Geometry Awareness Function.

        Args:
            x (torch.Tensor): Input feature tensor

        Returns:
            Tuple[torch.Tensor, torch.Tensor]: Embedded features and geometry probabilities
        """
        # Detect geometry characteristics
        geometry_probs = self.detect_geometry(x)

        # Embed in selected geometries
        embedded_x = self.embed_in_geometry(x, geometry_probs)

        return embedded_x, geometry_probs


def generate_euclidean_data(num_samples: int, input_dim: int) -> torch.Tensor:
    """
    Generate Euclidean data (standard normal distribution).
    """
    return torch.randn(num_samples, input_dim)


def generate_hyperbolic_data(num_samples: int, input_dim: int) -> torch.Tensor:
    """
    Generate data inside a Poincaré disk model (hyperbolic space).
    """
    theta = torch.rand(num_samples) * 2 * np.pi
    radius = torch.tanh(torch.rand(num_samples) * 5)  # Limit radius using tanh
    x = radius * torch.cos(theta)
    y = radius * torch.sin(theta)
    data = torch.zeros(num_samples, input_dim)
    data[:, 0] = x
    data[:, 1] = y
    return data


def generate_spherical_data(num_samples: int, input_dim: int) -> torch.Tensor:
    """
    Generate data on the surface of a unit sphere.
    """
    data = torch.randn(num_samples, input_dim)
    data = data / torch.norm(data, dim=1, keepdim=True)
    return data


def generate_fractal_data(num_samples: int, input_dim: int) -> torch.Tensor:
    """
    Generate fractal-like data (e.g., using the Mandelbrot set).
    For simplicity, we'll simulate fractal data with nested patterns.
    """
    data = torch.zeros(num_samples, input_dim)
    for i in range(num_samples):
        x = random.uniform(-2, 2)
        y = random.uniform(-2, 2)
        c = complex(x, y)
        z = 0
        iterations = 0
        max_iterations = 20
        while abs(z) <= 2 and iterations < max_iterations:
            z = z * z + c
            iterations += 1
        data[i, 0] = x
        data[i, 1] = y
        data[i, 2] = iterations / max_iterations  # Normalized iteration count
    return data


def create_dataset(num_samples_per_class: int, input_dim: int):
    """
    Create a dataset with samples from each geometry type.

    Returns:
        TensorDataset: Combined dataset
    """
    data_list = []
    labels_list = []

    # Euclidean
    euclidean_data = generate_euclidean_data(num_samples_per_class, input_dim)
    data_list.append(euclidean_data)
    labels_list.append(torch.zeros(num_samples_per_class, dtype=torch.long))  # Label 0

    # Hyperbolic
    hyperbolic_data = generate_hyperbolic_data(num_samples_per_class, input_dim)
    data_list.append(hyperbolic_data)
    labels_list.append(torch.ones(num_samples_per_class, dtype=torch.long))  # Label 1

    # Spherical
    spherical_data = generate_spherical_data(num_samples_per_class, input_dim)
    data_list.append(spherical_data)
    labels_list.append(torch.full((num_samples_per_class,), 2, dtype=torch.long))  # Label 2

    # Fractal
    fractal_data = generate_fractal_data(num_samples_per_class, input_dim)
    data_list.append(fractal_data)
    labels_list.append(torch.full((num_samples_per_class,), 3, dtype=torch.long))  # Label 3

    # Combine data and labels
    data = torch.cat(data_list, dim=0)
    labels = torch.cat(labels_list, dim=0)

    return TensorDataset(data, labels)


def train_model(model, dataloader, criterion, optimizer, device):
    """
    Train the model for one epoch.

    Args:
        model: The GeometryAwarenessFunction model.
        dataloader: DataLoader for training data.
        criterion: Loss function.
        optimizer: Optimizer.
        device: torch.device.
    """
    model.train()
    total_loss = 0
    correct = 0
    total_samples = 0

    for inputs, labels in dataloader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        _, geometry_probs = model(inputs)
        loss = criterion(geometry_probs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * inputs.size(0)
        _, predicted = geometry_probs.max(1)
        correct += predicted.eq(labels).sum().item()
        total_samples += inputs.size(0)

    avg_loss = total_loss / total_samples
    accuracy = correct / total_samples
    return avg_loss, accuracy


def evaluate_model(model, dataloader, criterion, device):
    """
    Evaluate the model.

    Args:
        model: The GeometryAwarenessFunction model.
        dataloader: DataLoader for validation/test data.
        criterion: Loss function.
        device: torch.device.
    """
    model.eval()
    total_loss = 0
    correct = 0
    total_samples = 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            _, geometry_probs = model(inputs)
            loss = criterion(geometry_probs, labels)

            total_loss += loss.item() * inputs.size(0)
            _, predicted = geometry_probs.max(1)
            correct += predicted.eq(labels).sum().item()
            total_samples += inputs.size(0)

    avg_loss = total_loss / total_samples
    accuracy = correct / total_samples
    return avg_loss, accuracy


def main():
    # Configuration
    input_dim = 16  # Reduced input dimension for simplicity
    batch_size = 64
    num_epochs = 10
    learning_rate = 0.001
    num_samples_per_class = 1000

    # Device configuration
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Create datasets
    dataset = create_dataset(num_samples_per_class, input_dim)
    num_total_samples = len(dataset)
    num_train = int(0.8 * num_total_samples)
    num_val = num_total_samples - num_train

    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [num_train, num_val])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    # Initialize the model
    model = GeometryAwarenessFunction(input_dim, hidden_dim=32).to(device)

    # Loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # Training loop
    for epoch in range(num_epochs):
        train_loss, train_accuracy = train_model(model, train_loader, criterion, optimizer, device)
        val_loss, val_accuracy = evaluate_model(model, val_loader, criterion, device)

        print(f"Epoch [{epoch + 1}/{num_epochs}] "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f} "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f}")

    # Final evaluation
    print("\nFinal Evaluation on Validation Set:")
    val_loss, val_accuracy = evaluate_model(model, val_loader, criterion, device)
    print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")


if __name__ == "__main__":
    main()

Epoch [1/10] Train Loss: 1.3746, Train Acc: 0.3594 Val Loss: 1.3399, Val Acc: 0.4813
Epoch [2/10] Train Loss: 1.3097, Train Acc: 0.4050 Val Loss: 1.2743, Val Acc: 0.4650
Epoch [3/10] Train Loss: 1.2582, Train Acc: 0.4631 Val Loss: 1.2235, Val Acc: 0.5563
Epoch [4/10] Train Loss: 1.2177, Train Acc: 0.5447 Val Loss: 1.1860, Val Acc: 0.5737
Epoch [5/10] Train Loss: 1.1856, Train Acc: 0.5525 Val Loss: 1.1549, Val Acc: 0.6000
Epoch [6/10] Train Loss: 1.1627, Train Acc: 0.5806 Val Loss: 1.1295, Val Acc: 0.6388
Epoch [7/10] Train Loss: 1.1434, Train Acc: 0.6325 Val Loss: 1.1094, Val Acc: 0.7025
Epoch [8/10] Train Loss: 1.1268, Train Acc: 0.6603 Val Loss: 1.0913, Val Acc: 0.7375
Epoch [9/10] Train Loss: 1.1091, Train Acc: 0.7025 Val Loss: 1.0748, Val Acc: 0.7700
Epoch [10/10] Train Loss: 1.0911, Train Acc: 0.7459 Val Loss: 1.0589, Val Acc: 0.8050

Final Evaluation on Validation Set:
Validation Loss: 1.0589, Validation Accuracy: 0.8050


In [ ]:
def test_geometry_awareness():
    """
    Demonstration and basic testing of the Geometry Awareness Function.
    """
    # Simulated input dimensions and batch size
    input_dim = 256
    batch_size = 64

    # Device configuration
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Initialize the Geometry Awareness Function
    geo_aware = GeometryAwarenessFunction(input_dim).to(device)

    # Generate random input data
    sample_input = torch.randn(batch_size, input_dim).to(device)

    # Process through Geometry Awareness Function
    geo_aware.eval()  # Set model to evaluation mode
    with torch.no_grad():
        embedded_features, geometry_probs = geo_aware(sample_input)

    print("Input Shape:", sample_input.shape)
    print("Embedded Features Shape:", embedded_features.shape)
    print("Geometry Probabilities (averaged over batch):")
    avg_geometry_probs = geometry_probs.mean(0)
    for geom, prob in zip(geo_aware.GEOMETRY_TYPES, avg_geometry_probs):
        print(f"{geom.capitalize()}: {prob.item():.4f}")


if __name__ == "__main__":
    test_geometry_awareness()

Input Shape: torch.Size([64, 256])
Embedded Features Shape: torch.Size([64, 256])
Geometry Probabilities (averaged over batch):
Euclidean: 0.2059
Hyperbolic: 0.2789
Spherical: 0.2439
Fractal: 0.2714
